In [15]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/7aad80ab3fe4ad5ffeabffb7c30bcb70edb218bd/01_data/processed/df_cleaned.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [16]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42645, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      30
negative               0
positive               0
neutral                0
compound               0
cyberbullying_type     0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [17]:
PROFANITY_WORDS = {
    'fuck', 'fucking', 'fucked', 'fuckin',
    'shit', 'shitty',
    'bitch', 'bitches',
    'asshole', 'assholes',
    'bastard',
    'damn',
    'dick',
    'piss',
    'crap',
    'bullshit',
    'motherfucker',
    'idiot',
    'stupid',
    'moron',
    'retard',
    'retarded'
}

In [18]:
def extract_linguistic_features(text):

    text = str(text)

    # Basic tokens
    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)
    char_count = len(text)
    unique_words = len(set(words))

    # Profanity
    profanity_words = [
        word for word in words
        if word in PROFANITY_WORDS
    ]

    profanity_count = len(profanity_words)
    unique_profanity_count = len(set(profanity_words))

    profanity_ratio = (
        profanity_count / word_count
        if word_count > 0 else 0
    )

    profanity_presence = int(profanity_count > 0)

    # Second-person language
    second_person_words = {
        'you',
        'your',
        'yours',
        'yourself',
        'yourselves',
        'youre',
        'youve',
        'youll',
        'youd'
    }

    second_person_count = sum(
        word in second_person_words
        for word in words
    )

    # Punctuation
    exclamation_count = text.count('!')
    question_count = text.count('?')
    comma_count = text.count(',')
    period_count = text.count('.')

    repeated_exclamation_count = len(
        re.findall(r'!{2,}', text)
    )

    repeated_question_count = len(
        re.findall(r'\?{2,}', text)
    )

    repeated_punctuation_count = len(
        re.findall(r'[!?]{2,}', text)
    )

    # Mentions / Hashtags
    mention_count = len(
        re.findall(r'@\w+', text)
    )

    hashtag_count = len(
        re.findall(r'#\w+', text)
    )

    # Return features
    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_count': unique_words,

        'profanity_count': profanity_count,
        'unique_profanity_count': unique_profanity_count,
        'profanity_ratio': profanity_ratio,
        'profanity_presence': profanity_presence,

        'second_person_count': second_person_count,

        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'comma_count': comma_count,
        'period_count': period_count,

        'repeated_exclamation_count': repeated_exclamation_count,
        'repeated_question_count': repeated_question_count,
        'repeated_punctuation_count': repeated_punctuation_count,

        'mention_count': mention_count,
        'hashtag_count': hashtag_count
    })


In [19]:
import re

In [20]:
linguistic_features = df['tweet_text'].apply(
    extract_linguistic_features
)

df = pd.concat(
    [df, linguistic_features],
    axis=1
)

In [21]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [22]:
print("Class-wise counts:")
for class_name, count in df['cyberbullying_type'].value_counts().items():
    print(f"{class_name}: {count}")

Class-wise counts:
religion: 7964
age: 7957
gender: 7491
ethnicity: 7427
not_cyberbullying: 6445
other_cyberbullying: 5361


In [23]:
print(df.columns.tolist())

['tweet_text', 'cyberbullying_type', 'clean_tweet', 'lemmatized_tweet', 'sentiment_scores', 'negative', 'positive', 'neutral', 'compound', 'word_count', 'char_count', 'unique_word_count', 'profanity_count', 'unique_profanity_count', 'profanity_ratio', 'profanity_presence', 'second_person_count', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'repeated_exclamation_count', 'repeated_question_count', 'repeated_punctuation_count', 'mention_count', 'hashtag_count']


In [24]:

# ============================================================
# WORD TF-IDF + SENTIMENT + BEHAVIORAL FEATURES
# Models: Logistic Regression and Linear SVM
# TF-IDF: 10,000 features, n-gram range (1, 2)
# Class imbalance: Balanced class weights
# Maximum iterations: 10,000
# ============================================================

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone

# ------------------------------------------------------------
# 1. SELECT FEATURES
# ------------------------------------------------------------

text_col = "lemmatized_tweet"

numeric_features = [
    "negative",
    "positive",
    "neutral",
    "compound",
    "second_person_count",
    "mention_count",
    "hashtag_count",
    "profanity_count",
    "unique_profanity_count",
    "profanity_ratio",
    "profanity_presence"
]

target_col = "cyberbullying_type"

df_exp = df[
    [text_col] + numeric_features + [target_col]
].copy()

# Remove rows without target labels
df_exp = df_exp.dropna(subset=[target_col])

# Text preprocessing
X_text = df_exp[text_col].fillna("").astype(str)

# Convert numeric features safely
X_numeric = (
    df_exp[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(
    df_exp[target_col].astype(str)
)

print("Classes:", list(label_encoder.classes_))
print("Dataset shape:", df_exp.shape)

# ------------------------------------------------------------
# 2. STRATIFIED TRAIN-TEST SPLIT
# ------------------------------------------------------------

(
    X_text_train,
    X_text_test,
    X_num_train,
    X_num_test,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Sparse numeric feature matrices
N_train = csr_matrix(
    X_num_train.to_numpy(dtype=float)
)
N_test = csr_matrix(
    X_num_test.to_numpy(dtype=float)
)

print("\nTraining samples:", len(y_train))
print("Testing samples :", len(y_test))

# ------------------------------------------------------------
# 3. WORD TF-IDF
# Fit ONLY on training text
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)

W_train = tfidf.fit_transform(X_text_train)
W_test = tfidf.transform(X_text_test)

print("\nWord TF-IDF features:", W_train.shape[1])

# ------------------------------------------------------------
# 4. COMBINE TEXT + NUMERIC FEATURES
# ------------------------------------------------------------

X_train = hstack(
    [W_train, N_train],
    format="csr"
)

X_test = hstack(
    [W_test, N_test],
    format="csr"
)

print("Combined training shape:", X_train.shape)
print("Combined testing shape :", X_test.shape)

# ------------------------------------------------------------
# 5. MODELS
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=10000,
        C=1.0,
        solver="lbfgs",
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        max_iter=10000,
        C=0.1,
        random_state=42
    )
}

# ------------------------------------------------------------
# 6. TRAIN + EVALUATE
# ------------------------------------------------------------

results = []
reports = {}
predictions = {}
confusion_matrices = {}
fitted_models = {}

for model_name, model_template in models.items():

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    model = clone(model_template)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    fitted_models[model_name] = model
    predictions[model_name] = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )
    weighted_f1 = f1_score(
        y_test, y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Weighted-F1": weighted_f1
    })

    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Macro-F1    : {macro_f1:.4f}")
    print(f"Weighted-F1 : {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    reports[model_name] = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

# ------------------------------------------------------------
# 7. COMPARISON TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "Macro-F1",
        ascending=False
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 8. NOT-CYBERBULLYING ANALYSIS
# ------------------------------------------------------------

focus_class = "not_cyberbullying"

if focus_class in label_encoder.classes_:

    print("\nNOT-CYBERBULLYING CLASS PERFORMANCE")
    print("-" * 65)

    for model_name, report in reports.items():

        metrics = report[focus_class]

        print(
            f"{model_name}: "
            f"Precision={metrics['precision']:.4f}, "
            f"Recall={metrics['recall']:.4f}, "
            f"F1={metrics['f1-score']:.4f}"
        )

# ------------------------------------------------------------
# 9. CONFUSION MATRICES
# ------------------------------------------------------------

for model_name, cm in confusion_matrices.items():

    print(f"\nConfusion matrix: {model_name}")
    print(pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    ))


Classes: ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']
Dataset shape: (42645, 13)

Training samples: 29851
Testing samples : 12794

Word TF-IDF features: 10000
Combined training shape: (29851, 10011)
Combined testing shape : (12794, 10011)

Logistic Regression
Accuracy    : 0.8499
Macro-F1    : 0.8332
Weighted-F1 : 0.8524

Classification Report:
                     precision    recall  f1-score   support

                age       0.96      0.97      0.96      2387
          ethnicity       0.98      0.98      0.98      2228
             gender       0.93      0.84      0.88      2248
  not_cyberbullying       0.60      0.61      0.61      1934
other_cyberbullying       0.57      0.65      0.61      1608
           religion       0.96      0.95      0.95      2389

           accuracy                           0.85     12794
          macro avg       0.84      0.83      0.83     12794
       weighted avg       0.86      0.85      0.85     12794

,Model,Accuracy,Macro-F1,Weighted-F1
0,Linear SVM,0.8524,0.8334,0.8527
1,Logistic Regression,0.8499,0.8332,0.8524



NOT-CYBERBULLYING CLASS PERFORMANCE
-----------------------------------------------------------------
Logistic Regression: Precision=0.6041, Recall=0.6075, F1=0.6058
Linear SVM: Precision=0.6196, Recall=0.5879, F1=0.6033

Confusion matrix: Logistic Regression
                      age  ethnicity  gender  not_cyberbullying  \
age                  2313          3       5                 40   
ethnicity               3       2184       3                 11   
gender                  8          7    1888                185   
not_cyberbullying      60         18      67               1175   
other_cyberbullying    23         17      57                448   
religion                1          2       6                 86   

                     other_cyberbullying  religion  
age                                   26         0  
ethnicity                             24         3  
gender                               153         7  
not_cyberbullying                    546        68  
othe

In [25]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [26]:

# ============================================================
# EXPERIMENT 2 — MODEL COMPARISON + HYPERPARAMETER TUNING
# Objective: Compare 5 ML/DL models using 3-fold CV Macro-F1
# Input: Existing TF-IDF + numeric feature matrices
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    ParameterGrid
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# 1. Validate existing notebook variables
# ------------------------------------------------------------

required_variables = [
    "X_train", "X_test", "y_train", "y_test", "label_encoder"
]

missing_variables = [
    variable for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}. "
        "Run your baseline preprocessing and train/test split cells first."
    )

if X_train.shape[0] != len(y_train):
    raise ValueError("X_train and y_train have different sample counts.")

if X_test.shape[0] != len(y_test):
    raise ValueError("X_test and y_test have different sample counts.")

if X_train.shape[1] != X_test.shape[1]:
    raise ValueError("X_train and X_test have different feature counts.")

# XGBoost multiclass classification expects encoded labels 0, 1, ..., K-1.
expected_labels = np.arange(len(label_encoder.classes_))
actual_labels = np.sort(np.unique(np.asarray(y_train)))

if not np.array_equal(actual_labels, expected_labels):
    raise ValueError(
        "Labels must be encoded as consecutive integers starting at 0. "
        "Check that label_encoder was fitted on the full target column."
    )

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)
print("Classes       :", list(label_encoder.classes_))

# ------------------------------------------------------------
# 2. Define models and hyperparameter grids
# ------------------------------------------------------------

model_configs = {
    "1. XGBoost": {
        "model": XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8]
        }
    },

    "2. SGD Classifier": {
        "model": SGDClassifier(
            class_weight="balanced",
            max_iter=10000,
            tol=1e-3,
            random_state=42
        ),
        "params": {
            "loss": ["hinge", "log_loss"],
            "alpha": [1e-4, 1e-3, 1e-2],
            "penalty": ["l2"]
        }
    },

    "3. Random Forest": {
        "model": RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "4. Extra Trees": {
        "model": ExtraTreesClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "5. MLP Neural Network": {
        "model": Pipeline([
            ("scaler", MaxAbsScaler()),
            ("mlp", MLPClassifier(
                solver="adam",
                activation="relu",
                batch_size=128,
                max_iter=100,
                early_stopping=True,
                n_iter_no_change=8,
                random_state=42
            ))
        ]),
        "params": {
            "mlp__hidden_layer_sizes": [(64,), (128,)],
            "mlp__alpha": [1e-4, 1e-3]
        }
    }
}

# ------------------------------------------------------------
# 3. Configure stratified cross-validation
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

scoring_metric = "f1_macro"

# ------------------------------------------------------------
# 4. Prepare result containers
# ------------------------------------------------------------

tuned_results = []
tuned_models = {}
tuned_predictions = {}
tuned_reports = {}
tuned_confusion_matrices = {}
grid_search_objects = {}

# Explicit class order for reports and confusion matrices
class_labels = np.arange(len(label_encoder.classes_))
class_names = [str(name) for name in label_encoder.classes_]

# ------------------------------------------------------------
# 5. Show estimated computational workload
# ------------------------------------------------------------

total_cv_fits = 0

for config in model_configs.values():
    combinations = len(ParameterGrid(config["params"]))
    total_cv_fits += combinations * cv_strategy.get_n_splits()

total_final_refits = len(model_configs)

print(f"\nModels to tune          : {len(model_configs)}")
print(f"Total CV fits           : {total_cv_fits}")
print(f"Final best-model refits : {total_final_refits}")
print(f"Estimated total fits    : {total_cv_fits + total_final_refits}")
print(f"CV scoring metric       : {scoring_metric}")
print("\nNote: actual runtime depends on your machine and dataset size.")

# ------------------------------------------------------------
# 6. Run grid search for each model
# ------------------------------------------------------------

for model_name, config in tqdm(
    model_configs.items(),
    total=len(model_configs),
    desc="Overall Grid Search",
    unit="model"
):
    print("\n" + "=" * 75)
    print(f"MODEL: {model_name}")
    print("=" * 75)

    model = config["model"]
    param_grid = config["params"]

    n_combinations = len(ParameterGrid(param_grid))
    n_splits = cv_strategy.get_n_splits(X_train, y_train)
    expected_cv_fits = n_combinations * n_splits

    print("Parameter combinations:", n_combinations)
    print("CV folds               :", n_splits)
    print("CV fits                :", expected_cv_fits)
    print("Final refit            : 1")

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring_metric,
        cv=cv_strategy,
        refit=True,
        n_jobs=1,
        verbose=2,
        error_score="raise",
        return_train_score=False
    )

    start_time = time.time()

    # Fit only on training data; GridSearchCV handles CV internally.
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time

    best_model = search.best_estimator_

    # Final evaluation on the held-out test set
    y_pred = best_model.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_macro_f1 = f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )
    test_weighted_f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )

    # Full class-wise report
    report_dict = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        zero_division=0
    )

    # Confusion matrix with consistent class order
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=class_labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual: {name}" for name in class_names],
        columns=[f"Predicted: {name}" for name in class_names]
    )

    # Store all outputs
    tuned_models[model_name] = best_model
    tuned_predictions[model_name] = y_pred
    tuned_reports[model_name] = report_dict
    tuned_confusion_matrices[model_name] = cm_df
    grid_search_objects[model_name] = search

    tuned_results.append({
        "Model": model_name,
        "Best CV Macro-F1": search.best_score_,
        "Test Accuracy": test_accuracy,
        "Test Macro-F1": test_macro_f1,
        "Test Weighted-F1": test_weighted_f1,
        "Best Parameters": search.best_params_,
        "CV Fits": expected_cv_fits,
        "Runtime (minutes)": elapsed_time / 60
    })

    print("\nBest parameters:")
    print(search.best_params_)

    print(f"\nBest CV Macro-F1 : {search.best_score_:.4f}")
    print(f"Test Accuracy    : {test_accuracy:.4f}")
    print(f"Test Macro-F1    : {test_macro_f1:.4f}")
    print(f"Test Weighted-F1 : {test_weighted_f1:.4f}")
    print(f"Runtime          : {elapsed_time / 60:.2f} minutes")

    print("\nClassification Report:")
    print(report_text)

    print("\nConfusion Matrix:")
    display(cm_df)

# ------------------------------------------------------------
# 7. Compare models
# ------------------------------------------------------------

tuned_results_df = pd.DataFrame(tuned_results)

# Rank using CV Macro-F1, not held-out test performance.
tuned_results_df = tuned_results_df.sort_values(
    by="Best CV Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)

display(
    tuned_results_df[
        [
            "Model",
            "Best CV Macro-F1",
            "Test Accuracy",
            "Test Macro-F1",
            "Test Weighted-F1",
            "Runtime (minutes)"
        ]
    ].round(4)
)

print("\nBest hyperparameters for each model:")
for _, row in tuned_results_df.iterrows():
    print(f"\n{row['Model']}")
    print(row["Best Parameters"])

# ------------------------------------------------------------
# 8. Inspect Not-Cyberbullying performance
# ------------------------------------------------------------

not_cb_class = "not_cyberbullying"

if not_cb_class in class_names:
    print("\n" + "=" * 90)
    print("NOT-CYBERBULLYING CLASS PERFORMANCE")
    print("=" * 90)

    not_cb_results = []

    for model_name, report in tuned_reports.items():
        class_metrics = report.get(not_cb_class, {})

        not_cb_results.append({
            "Model": model_name,
            "Not-CB Precision": class_metrics.get("precision", np.nan),
            "Not-CB Recall": class_metrics.get("recall", np.nan),
            "Not-CB F1": class_metrics.get("f1-score", np.nan),
            "Not-CB Support": class_metrics.get("support", np.nan)
        })

    not_cb_results_df = pd.DataFrame(not_cb_results)

    not_cb_results_df = not_cb_results_df.sort_values(
        by="Not-CB F1",
        ascending=False
    ).reset_index(drop=True)

    display(not_cb_results_df.round(4))

else:
    print(
        f"\n'{not_cb_class}' was not found in label_encoder.classes_. "
        "Check the original target labels."
    )

print("\nExperiment 2 complete.")


Training shape: (29851, 10011)
Testing shape : (12794, 10011)
Classes       : ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']

Models to tune          : 5
Total CV fits           : 102
Final best-model refits : 5
Estimated total fits    : 107
CV scoring metric       : f1_macro

Note: actual runtime depends on your machine and dataset size.


Overall Grid Search:   0%|          | 0/5 [00:00<?, ?model/s]


MODEL: 1. XGBoost
Parameter combinations: 8
CV folds               : 3
CV fits                : 24
Final refit            : 1
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  27.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  27.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  26.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  49.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  50.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=  50.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=100, subsample=0.8; total time= 1.6min
[

,Predicted: age,Predicted: ethnicity,Predicted: gender,Predicted: not_cyberbullying,Predicted: other_cyberbullying,Predicted: religion
Actual: age,2313,2,1,44,27,0
Actual: ethnicity,1,2193,4,10,19,1
Actual: gender,2,2,1903,237,102,2
Actual: not_cyberbullying,17,7,77,1362,409,62
Actual: other_cyberbullying,8,6,67,606,917,4
Actual: religion,2,1,3,82,29,2272



MODEL: 2. SGD Classifier
Parameter combinations: 6
CV folds               : 3
CV fits                : 18
Final refit            : 1
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.5s
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.6s
[CV] END ...............alpha=0.0001, loss=hinge, penalty=l2; total time=   0.5s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.5s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.6s
[CV] END ............alpha=0.0001, loss=log_loss, penalty=l2; total time=   0.6s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END ................alpha=0.001, loss=hinge, penalty=l2; total time=   0.3s
[CV] END .............alpha=0.001, loss=log_loss, penalty=l2; total time=   0

...


FINAL MODEL COMPARISON


,Model,Best CV Macro-F1,Test Accuracy,Test Macro-F1,Test Weighted-F1,Runtime (minutes)
0,1. XGBoost,0.8296,0.8567,0.8397,0.8597,35.1257
1,2. SGD Classifier,0.8202,0.8477,0.8271,0.8464,0.1312
2,3. Random Forest,0.8196,0.8426,0.8267,0.8471,2.9685
3,5. MLP Neural Network,0.8097,0.8427,0.8216,0.8420,12.3594
4,4. Extra Trees,0.7904,0.8237,0.7958,0.8212,1.7436



Best hyperparameters for each model:

1. XGBoost
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}

2. SGD Classifier
{'alpha': 0.0001, 'loss': 'hinge', 'penalty': 'l2'}

3. Random Forest
{'max_depth': 40, 'min_samples_leaf': 1, 'n_estimators': 200}

5. MLP Neural Network
{'mlp__alpha': 0.001, 'mlp__hidden_layer_sizes': (128,)}

4. Extra Trees
{'max_depth': 40, 'min_samples_leaf': 1, 'n_estimators': 200}

NOT-CYBERBULLYING CLASS PERFORMANCE


,Model,Not-CB Precision,Not-CB Recall,Not-CB F1,Not-CB Support
0,1. XGBoost,0.5818,0.7042,0.6372,1934.0
1,4. Extra Trees,0.5182,0.7746,0.6209,1934.0
2,3. Random Forest,0.5795,0.6334,0.6052,1934.0
3,5. MLP Neural Network,0.5947,0.6138,0.6041,1934.0
4,2. SGD Classifier,0.6151,0.5760,0.5949,1934.0



Experiment 2 complete.
